# 04 · Gold — feature table et labels d'attrition
Population mensuelle des clients en relation, features calculées uniquement avec le passé,
stockées dans une feature table Unity Catalog. Labels : clôture dans les 90 jours, seulement pour
les dates dont la fenêtre est entièrement observée.

In [0]:
%pip install -q databricks-feature-engineering
%restart_python

In [0]:
import os, sys, datetime as dt
sys.path.append(os.path.abspath("../src"))
import pandas as pd
from pyspark.sql import functions as F
from databricks.feature_engineering import FeatureEngineeringClient
from churn.dbx import get_config
from churn.config import KEYS, TRAIN_SNAPSHOTS, month_starts, latest_snapshot
from churn import transforms as T

cfg = get_config(dbutils)
fe = FeatureEngineeringClient()

ops = spark.table(cfg.t("silver_operations"))
interactions = spark.table(cfg.t("silver_interactions"))
clients = spark.table(cfg.t("silver_clients"))
clotures = spark.table(cfg.t("silver_clotures"))

max_date = ops.agg(F.max("op_date")).first()[0]            # dernier jour de données
last_snap = latest_snapshot(max_date)                       # 1er du mois suivant
snap_dates = month_starts(TRAIN_SNAPSHOTS[0], str(last_snap))
print(f"Données jusqu'au {max_date} → {len(snap_dates)} dates d'observation : {snap_dates[0]} … {last_snap}")

In [0]:
snapshots = spark.createDataFrame(pd.DataFrame({"snapshot_date": snap_dates}))
features = T.build_features(ops, interactions, clients, clotures, snapshots)

ft_name = cfg.t("client_features")
if not spark.catalog.tableExists(ft_name):
    # 1re fois : création de la feature table avec clé primaire + colonne temporelle
    fe.create_table(
        name=ft_name,
        primary_keys=KEYS,                       # (client_id, snapshot_date)
        timeseries_columns="snapshot_date",      # active la jointure point-in-time
        df=features,
        description="Features client (activité, domiciliation, fuite concurrence, irritants, digital, "
                    "équipement) au 1er de chaque mois — aucune donnée nominative")
else:
    # fois suivantes : mise à jour (upsert sur la clé primaire)
    fe.write_table(name=ft_name, df=features, mode="merge")

print(ft_name, spark.table(ft_name).count(), "lignes")

In [0]:
matured = T.matured_snapshots(snap_dates, max_date + dt.timedelta(days=1), cfg.churn_horizon_days)
print(f"Dates avec label connu : {matured[0]} … {matured[-1]} ({len(matured)} dates)")

pop = spark.table(ft_name).where(F.col("snapshot_date").isin(matured)).select(*KEYS)
labels = T.build_labels(clotures, pop, cfg.churn_horizon_days)
labels.write.mode("overwrite").saveAsTable(cfg.t("attrition_labels"))

display(spark.table(cfg.t("attrition_labels")).groupBy("snapshot_date")
        .agg(F.count("*").alias("clients_en_relation"),
             F.round(F.avg("churn"), 4).alias("taux_attrition_90j"))
        .orderBy("snapshot_date"))